## 手写transformer


![transformer](../images/transformer.png)  
## 代码更新方向
### 1.mask掩码，现在我使用的长度掩码，我需要将其更新为工业级的 随机掩码
### 2.针对于tokenize_nmt_clean（词元化），我现在是按照单词划分，我需要将其更新为按照“子词”划分
### 3.新的数据清洗方案-----我目前这份数据清洗方案，是由Codex提供的，我虽然看得懂，但这不是我写的，所以，我需要将其更新为工业级的数据清洗方案

In [ ]:

import torch

if torch.cuda.is_available():
    print("✅ GPU 可用:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("⚠️ 使用 CPU")
    device = torch.device("cpu")


✅ GPU 可用: NVIDIA GeForce RTX 4090


In [196]:
# 导入必要的库
import math
import torch
from torch import nn
import collections
# 用于封装数据集
from torch.utils.data import DataLoader,TensorDataset
# 用于读取数据集
import os
import hashlib
import urllib.request
import zipfile
import re
import random


#  1.Embeding+位置编码(PositionalEncoding)
![位置编码公式.png](../images/PositionalEncoding.png)

In [197]:
# 位置编码
class PositionalEncoding(nn.Module):
    """
    用于生成位置编码
    1. 生成位置编码矩阵，包含每个位置的编码信息
    2. 将输入的嵌入向量与位置编码相加，得到包含位置信息的输入表示
    3. 返回最终的输出

    位置编码的引入使得模型能够捕捉输入序列中元素之间的相对和绝对位置信息，从而更好地理解序列数据。
    现在已经不在使用这种方法了（除了教学）
    """
    def __init__(self,num_hiddens,dropout,maxlen=1000,**kwargs):
        """
        num_hiddens: 模型的隐藏层维度 表示位置编码矩阵的列数，也就是每个位置编码的维度
        dropout: dropout 的概率
        max_lens: 位置编码的最大长度，通常设置为一个较大的值, 表示位置编码矩阵的行数，
                    但一般用不了那么长,所以有:X=X+self.P[:,0:X.shape[1],:].to(X.device)
                    来限制位置编码矩阵的行数
        """
        super(PositionalEncoding,self).__init__(**kwargs)
        self.P=torch.zeros(1,maxlen,num_hiddens)
        X=torch.arange(maxlen,dtype=torch.float32).reshape(-1,1)/torch.pow(10000,
                    torch.arange(0,num_hiddens,2,dtype=torch.float32)/num_hiddens)
        self.P[:,:,0::2]=torch.sin(X)
        self.P[:,:,1::2]=torch.cos(X)
        self.dropout=nn.Dropout(dropout)
    def forward(self,X):
        X=X+self.P[:,0:X.shape[1],:].to(X.device)
        # 这一步涉及到广播机制，
        # P 的有效形状形状为 (1, seq_len, num_hiddens)，
        # X 的形状为 (batch_size, seq_len, num_hiddens)
        return self.dropout(X)

# 2.多头注意力(Multi-Head Attention)
## 关于多头注意力中数据形状的变化
![多头原理.jpg](../images/MultiHeadAttention1.jpg)
![分头和合并.jpg](../images/MultiHeadAttention2.jpg)
![valid_len变化.jpg](../images/MultiHeadAttention3.jpg)

In [198]:
# 多头注意力
def transpose_qkv(X,num_heads):
    """
    用于多头注意力机制的分头处理
    X的形状为 (batch_size, seq_len, num_hiddens)
    num_heads为多头注意力的头数
    返回的形状为 (batch_size * num_heads, seq_len, num_hiddens // num_heads)
    """
    X=X.reshape(X.shape[0],X.shape[1],num_heads,-1)
    X=X.permute(0,2,1,3)
    return X.reshape(-1,X.shape[2],X.shape[3])
def transpose_out(X,num_heads):
    """
    用于多头注意力机制的恢复处理
    X的形状为 (batch_size * num_heads, seq_len, num_hiddens // num_heads)
    num_heads为多头注意力的头数
    返回的形状为 (batch_size, seq_len, num_hiddens)
    """
    X=X.reshape(-1,num_heads,X.shape[1],X.shape[2])
    X=X.permute(0,2,1,3)
    return X.reshape(X.shape[0],X.shape[1],-1)

def mask_softmax(X,valid_lens=None):
    """
    实现所谓的掩码的处理，相当于遮蔽住不想让模型看到的部分，
    但我这部分的实现为“长度截断式”，仅用于教学，实际工业生产中不使用。
    X 的形状为 (batch_size * num_heads, query_len, key_len)
    valid_lens的形状为 (batch_size, seq_len)或(batch_size*num_heads, )
    返回的形状为 (batch_size, seq_len, num_hiddens)
    valid_lens为一个形状为 (batch_size, seq_len) 的张量，
    valid_lens[i] 表示第 i 个样本的序列长度，小于 valid_lens[i] 的部分将被截断，
    大于 valid_lens[i] 的部分将被填充。
    """
    if valid_lens is None:
        return nn.functional.softmax(X,dim=-1)
        # nn.functional.softmax(X,dim=-1) 与 nn.softmax的区别
    else:
        shape=X.shape
        # print(f"valid_lens_dim={valid_lens.dim()}")
        # print(f"valid_len={valid_lens}")
        if valid_lens.dim()==1:
            valid_lens=torch.repeat_interleave(valid_lens,X.shape[1])
        else:
            valid_lens=valid_lens.reshape(-1)
        X=X.reshape(-1,X.shape[-1])
        maxlen=X.shape[1]
        mask=torch.arange(maxlen,dtype=torch.float32,device=X.device)[None,:]<valid_lens[:,None]
        # None 在 PyTorch（以及 NumPy）的张量索引中具有特殊含义——它表示“在该位置插入一个大小为 1 的新维度”（即 unsqueeze 操作）
        # mask=torch.arange(maxlen,dtype=torch.float32,device=X.device).reshape(1,-1)
        # valid_lens=valid_lens.reshape(-1,1)
        # mask=mask<valid_lens    # 广播
        X[~mask]=-1e6

        return nn.functional.softmax(X.reshape(shape),dim=-1)


class DotProductAttention(nn.Module):
    """
    点积注意力
    q的形状为 (batch_size, seq_len, num_hiddens)
    k的形状为 (batch_size, seq_len, num_hiddens)
    v的形状为 (batch_size, seq_len, num_hiddens)
    valid_lens的形状为 (batch_size, seq_len)或(batch_size*num_heads, )
    返回的形状为 (batch_size, seq_len, num_hiddens)
    """
    def __init__(self,dropout,**kwargs):
        super(DotProductAttention,self).__init__(**kwargs)
        self.dropout=nn.Dropout(dropout)
    def forward(self,q,k,v,valid_lens=None):
        scores=q@k.transpose(1,2)/math.sqrt(q.shape[-1])
        self.attention_weights=mask_softmax(scores,valid_lens)
        return self.dropout(self.attention_weights)@v




class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    1. 将输入的查询、键、值分别通过线性变换得到多个头的查询、键、值
    2. 对每个头的查询、键、值进行缩放点积注意力计算，得到每个头的输出
    3. 将所有头的输出拼接起来，并通过线性变换得到最终的输出
    4. 返回最终的输出和注意力权重
    5. 注意力权重的形状为 (batch_size, num_heads, query_length, key_length)
    多头注意力机制通过引入多个注意力头，使得模型能够同时关注输入的不同部分，从而捕捉更丰富的特征信息。
    """
    def __init__(self,q_size,k_size,v_size,num_hiddens,num_heads,dropout,**kwargs):
        """
            q_size,k_size,v_size: 输入的查询、键、值的维数,但其一般就是隐藏层维数(num_hiddens),
            num_hiddens: 隐藏层的维数
            num_heads: 头数
            dropout: 丢弃概率
        """
        super(MultiHeadAttention,self).__init__(**kwargs)
        self.num_heads=num_heads
        self.w_q=nn.Linear(q_size,num_hiddens,bias=False)
        self.w_k=nn.Linear(k_size,num_hiddens,bias=False)
        self.w_v=nn.Linear(v_size,num_hiddens,bias=False)
        self.w_o=nn.Linear(num_hiddens,num_hiddens,bias=False)
        self.attention=DotProductAttention(dropout)
    def forward(self,q,k,v,valid_lens=None):
        # 先分头
        q=transpose_qkv(self.w_q(q),self.num_heads)
        k=transpose_qkv(self.w_k(k),self.num_heads)
        v=transpose_qkv(self.w_v(v),self.num_heads)

        # 操作一下有效值范围
        if valid_lens is not None:
            valid_lens=torch.repeat_interleave(valid_lens,self.num_heads,dim=0)

        out=self.attention(q,k,v,valid_lens)

        out_put=transpose_out(out,self.num_heads)
        return self.w_o(out_put)

# 3.层归一化(layer_normalization)
![层归一化和batch归一化.png](../images/层归一化和batch归一化.png)
![层归一化中数据的形状.jpg](../images/Layer_norm.jpg)
输入该层的数据形状为(batch_size, seq_len,, num_hiddens)。每一个batch_size代表一句话，每句话里面有多个词。一个词代表一行数据，每句话的词个数不同。为了统一词数为seq_len ,需要对部分句子填充几个词，使其句子长度为seq_len。，但这一步实在Embedding层完成。
最后导致的结果是，层归一化的输入数据形状为(batch_size, seq_len, num_hiddens)。每个batch_size代表一句话，每句话里面有seq_len个词。一个词用num_hiddens个特征表示其蕴含的特征
层归一化，是在对单个词进行归一化处理，而不是对整个句子进行归一化处理。使得每一句话中的每一词减去该词的均值，除以该词的方差，使得该词的num_hiddens个特征向量都满足正太分布，最后通过可学习的缩放和平移参数调整，从而稳定训练过程中的数值分布。"

In [199]:
# 层归一化
class LayerNorm(nn.Module):
    """
    层归一化
    1. 计算输入的均值和标准差
    2. 对输入进行归一化处理，使其均值为0，标准差为1
    3. 使用可学习的参数 gamma 和 beta 对归一化后的结果进行缩放和平移
    4. 返回最终的输出

    与批量归一化不同，层归一化在每个样本的特征维度上进行归一化，而不是在批次维度上。
    这使得层归一化在处理变长序列或小批量数据时更有效。
    """
    def __init__(self,num_hiddens,eps=1e-3,**kwargs):
        super(LayerNorm,self).__init__(**kwargs)
        self.eps=eps
        self.gamma=nn.Parameter(torch.ones(1,num_hiddens))
        self.bate=nn.Parameter(torch.zeros(1,num_hiddens))
        # nn.Parameter 是 torch.Tensor 的子类，用于将张量注册为神经网络模块中的可学习参数。
    def forward(self,X):
        X_mean=X.mean(dim=-1,keepdim=True)
        # 沿着num_hiddens维度求mean，输出形状为(batcch_size,seq_size,1)
        X_var=X.var(dim=-1,unbiased=True,keepdim=True)
        # 沿着num_hiddens维度求var，输出形状为(batcch_size,seq_size,1)
        # unbiased=False 为有偏估计
        # std = sqrt(var + eps)  std为标准差，var为方差
        X=(X-X_mean)/torch.sqrt(X_var+self.eps)
        # 使用了广播的方式
        return self.gamma*X+self.bate

In [200]:

class AddNorm(nn.Module):
    """
    残差连接
    1. 将输入和子层的输出相加
    2. 对相加的结果进行层归一化处理
    3. 返回最终的输出

    残差连接的引入使得模型能够更好地捕捉输入特征之间的关系，同时避免了深层网络中常见的梯度消失问题。
    """
    def __init__(self,norm_shape,dropout,**kwargs):
        """
        norm_shape: 层归一化的参数,但其一般就是隐藏层维数(num_hiddens)
        dropout: 随机失活的概率
        """
        super(AddNorm,self).__init__(**kwargs)
        self.dropout=nn.Dropout(dropout)
        self.layer_norm=LayerNorm(norm_shape)
    def forward(self,X,Y):
        """X 是子层的输入，Y 是子层的输出"""
        return self.layer_norm(self.dropout(Y)+X)


In [201]:
# 前馈神经网络（FFN）
class PositionWiseFFN(nn.Module):
    """

    PyTorch 的 nn 模块基于 nn.Module 这个基类构建。
    所有神经网络组件（无论是单个线性层还是整个 ResNet）
    都是 nn.Module 的子类。

    位置前馈网络 (FFN)
    FFN 由两个全连接层组成。第一个全连接层的输出维度通常远大于输入维度，这叫升维度，是为了提高表达能力
    第二个全连接层的输出维度通常小于输入维度，这叫降维度，是为了减少参数数量，从而提高效率
    先升维度，再降维度的设计使得 FFN 能够捕捉输入特征之间的复杂关系，同时保持计算效率
    类似于，你先学习消化了很多知识（升维度），再将这些简要概括（降维度）

    """
    def __init__(self,num_hiddens,hidden_size,dropout,**kwargs):
        super(PositionWiseFFN,self).__init__(**kwargs)
        self.dense1=nn.Linear(num_hiddens,hidden_size)
        self.relu=nn.ReLU()
        self.dense2=nn.Linear(hidden_size,num_hiddens)
        self.dropout=nn.Dropout(dropout)
    def forward(self,X):
        return self.dropout(self.dense2(self.relu(self.dense1(X))))


In [202]:
# 编码器块
class EncoderBlack(nn.Module):
    """
    编码器块
    1. 包含一个多头注意力子层和一个位置前馈网络子层
    2. 每个子层后面都跟着一个残差连接和层归一化
    3. 在前向传播过程中，输入首先通过多头注意力子层进行处理，然后通过残差连接和层归一化得到第一个输出
    4. 第一个输出再通过位置前馈网络子层进行处理，然后通过残差连接和层归一化得到最终的输出
    5. 返回最终的输出

    编码器块是 Transformer 模型中的基本构建模块，通过堆叠多个编码器块，模型能够捕捉输入序列中更复杂的特征关系。
    """
    def __init__(self,q_size,k_size,v_size,num_hiddens,num_heads,ffn_hidden_size,dropout,**kwargs):
        """
        q_size,K_size,V_size: 输入的维度
        num_hiddens: 隐藏层的维度
        num_heads: 多头注意力的头数
        ffn_hidden_size: 前向传播隐藏层的维度
        dropout: 丢弃概率
        """
        super(EncoderBlack,self).__init__(*kwargs)
        self.attention=MultiHeadAttention(q_size,k_size,v_size,num_hiddens,num_heads,dropout)
        self.add_norm1=AddNorm(num_hiddens,dropout)
        self.ffn=PositionWiseFFN(num_hiddens,ffn_hidden_size,dropout)
        self.add_norm2=AddNorm(num_hiddens,dropout)
    def forward(self,X,valid_lens):
        """
        X: 输入序列
        valid_lens: encoder输入序列的有效长度
        """
        Y=self.add_norm1(X,self.attention(X,X,X,valid_lens))
        return self.add_norm2(Y,self.ffn(Y))


In [203]:
# 编码器
class TransformerEncoder(nn.Module):
    """
    Transformer 编码器
    1. 包含多个编码器块和一个位置编码层
    2. 在前向传播过程中，输入首先通过位置编码层进行处理，然后依次通过每个编码器块进行处理
    3. 返回最终的输出

    Transformer 编码器通过堆叠多个编码器块，能够捕捉输入序列中更复杂的特征关系，从而提高模型的表达能力。
    """
    def __init__(self,vocab_size,q_size,k_size,v_size,num_hiddens,num_heads,
                ffn_hidden_size,num_layer,dropout,**kwargs):
        """
        vocab_size: 词表大小
        q_size,k_size,v_size: 输入的维度
        num_hiddens: 隐藏层维度
        num_heads: 多头注意力的头数
        ffn_hidden_size: 前向传播隐藏层维度
        num_layer: 编码器块的数量
        dropout: 丢弃概率
        """
        super(TransformerEncoder,self).__init__(**kwargs)
        self.embedding=nn.Embedding(vocab_size,num_hiddens)
        self.pos_encoding=PositionalEncoding(num_hiddens,dropout)
        self.blks=nn.Sequential()
        for i in range (num_layer):
            self.blks.add_module("blk"+str(i),EncoderBlack(q_size,k_size,v_size,
                                                        num_hiddens,num_heads,ffn_hidden_size,dropout))

    def forward(self,X,valid_lens):
        """
        X: 输入序列
        valid_lens: encoder输入序列的有效长度
        """
        X=self.pos_encoding(self.embedding(X)*math.sqrt(X.shape[-1]))
        for blk in self.blks:
            X=blk(X,valid_lens)

        return X


In [204]:
# 解码器块
class Decoder(nn.Module):
    """
    解码器块
    1. 包含一个多头注意力子层、一个编码器-解码器注意力子层和一个位置前馈网络子层
    2. 每个子层后面都跟着一个残差连接和层归一化
    3. 在前向传播过程中，输入首先通过多头注意力子层进行处理，然后通过残差连接和层归一化得到第一个输出
    4. 第一个输出再通过编码器-解码器注意力子层进行处理，然后通过残差连接和层归一化得到第二个输出
    5. 第二个输出再通过位置前馈网络子层进行处理，然后通过残差连接和层归一化得到最终的输出
    6. 返回最终的输出

    解码器块是 Transformer 模型中的基本构建模块，通过堆叠多个解码器块，模型能够捕捉输入序列中更复杂的特征关系，从而提高模型的表达能力。
    """
    def __init__(self,q_size,k_size,v_size,num_hiddens,num_heads,
                ffn_hidden_size,i,dropout,**kwargs):
        super(Decoder,self).__init__(**kwargs)
        self.i=i
        self.attention1=MultiHeadAttention(q_size,k_size,v_size,num_hiddens,num_heads,dropout)
        self.add_norm1=AddNorm(num_hiddens,dropout)

        self.attention2=MultiHeadAttention(q_size,k_size,v_size,num_hiddens,num_heads,dropout)
        self.add_norm2=AddNorm(num_hiddens,dropout)

        self.ffn=PositionWiseFFN(num_hiddens,ffn_hidden_size,dropout)
        self.add_norm3=AddNorm(num_hiddens,dropout)

    def forward(self,X,valid_lens,state):
        """
        X: 输入序列，shape为(batch_size,num_steps,num_hiddens)
        valid_lens: decoder输入序列的有效长度，shape为(batch_size,num_steps)
        state: 状态，包含编码器的输出和编码器的有效长度
        state中的内容为:
        state[0] 存储编码器的输出
        state[1] 存储编码器的有效长度
        state[2] 存储解码器的状态，它包含多个元素，每个元素对应一个解码器块，存储着该块的输出表示
        """
        enc_out,enc_valid_lens=state[0],state[1]

        if state[2][self.i] is None:
            k_values=X
        else:
            k_values=torch.cat((state[2][self.i],X),dim=1)
        state[2][self.i]=k_values

        
            # 这一部分我根据李沐老师的代码做了更新
            # 添加了解码序列有效长度掩码，使得解码器不需要看填充(padding)的部分
        batch_size,num_steps,_=X.shape
        dec_valid_lens=torch.arange(1,num_steps+1,device=X.device).repeat(batch_size,1)
        if valid_lens is not None:
            if valid_lens.dim()==1:
                valid_lens=valid_lens.reshape(-1,1).repeat(1, num_steps)
            dec_valid_lens=torch.min(dec_valid_lens,valid_lens)
                # print("dec_valid_lens[0] =", dec_valid_lens[0])
                # print("valid_lens[0] =", valid_lens[0] if valid_lens is not None else None)


        X2=self.attention1(X,k_values,k_values,dec_valid_lens)
        Y=self.add_norm1(X,X2)
        Y2=self.attention2(Y,enc_out,enc_out,enc_valid_lens)
        Z=self.add_norm2(Y,Y2)
        return self.add_norm3(Z,self.ffn(Z))

In [205]:
# 解码器
class TransformerDecoder(nn.Module):
    """
    Transformer 解码器
    1. 包含多个解码器块和一个位置编码层
    2. 在前向传播过程中，输入首先通过位置编码层进行处理，然后依次通过每个解码器块进行处理
    3. 返回最终的输出

    Transformer 解码器通过堆叠多个解码器块，能够捕捉输入序列中更复杂的特征关系，从而提高模型的表达能力。
    """
    def __init__(self,vocab_size,q_size,k_size,v_size,num_hiddens,num_heads,
                ffn_hidden_size,num_layer,dropout,**kwargs):
        super(TransformerDecoder,self).__init__(**kwargs)
        self.num_hiddens=num_hiddens
        self.embedding=nn.Embedding(vocab_size,num_hiddens)
        self.pos_encoding=PositionalEncoding(num_hiddens,dropout)
        self.blks=nn.Sequential()
        for i in range(num_layer):
            self.blks.add_module("blk"+str(i),
                                Decoder(q_size,k_size,v_size,num_hiddens,
                                        num_heads,ffn_hidden_size,i,dropout))
        self.dense=nn.Linear(num_hiddens,vocab_size)

    def init_state(self,enc_output,enc_valid_lens,*args):
        return [enc_output,enc_valid_lens,[None]*len(self.blks)]
    def forward(self,X,valid_len,state):
        """
        X: 输入序列，shape=(batch_size,num_steps,num_hiddens)
        valid_lens: 解码器输入序列的有效长度，shape=(batch_size,)
        state: 解码器的状态，包含编码器的输出和编码器的有效长度
        """
        X=self.pos_encoding(self.embedding(X)*math.sqrt(self.num_hiddens))
        # X=self.pos_encoding(self.embedding(X)*math.sqrt(X.shape[-1]))
        # 这个位置一定要写成self_hiddens，因为推理时的输入，X.shape[-1]=1
        for blk in self.blks:
            X=blk(X,valid_len,state)
        return self.dense(X)

In [206]:
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, **kwargs):
        super(EncoderDecoder, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, enc_valid_lens=None, dec_valid_lens=None):
        """
        enc_X: 编码器输入序列, shape=(B, S)
        dec_X: 解码器输入序列, shape=(B, T)
        enc_valid_lens: 编码器输入有效长度, shape=(B,)
        dec_valid_lens: 解码器输入有效长度, shape=(B,) 或 None
        """
        enc_outputs = self.encoder(enc_X, enc_valid_lens)
        dec_state = self.decoder.init_state(enc_outputs, enc_valid_lens)
        dec_outputs= self.decoder(dec_X, dec_valid_lens, dec_state)
        return dec_outputs


# =========================================================
# 在我已有 Transformer 核心代码基础上补充：
# tokenizer + 数据处理 + 训练 + 推理
# =========================================================

In [207]:
# =========================================================
# 1) tokenizer 与词表
# =========================================================

In [ ]:
# 这一段主要是为了下载数据集，
# 但是因为网站访问的问题，我下载不了
# 所以我是手动下载的
# 数据集文件在my_code/data/fra-eng.zip,将其解压到my_code/data/fra-eng中
# about.txt和fra.txt都得在my_code/data/fra-eng中，否则代码找不到这两个文件
def sha1sum(filename, chunk_size=1 << 20):
    """计算文件 SHA1,用于校验下载完整性。"""
    h = hashlib.sha1()
    with open(filename, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def download_fra_eng(
    cache_dir="./data",
    filename="fra-eng.zip",
    expected_sha1="94646ad1522d915e7b0f9296181140edcf86a4f5",
    strict_sha1=False
):
    """
    下载 Tatoeba 英法数据集压缩包（manythings 版本）。
    strict_sha1=False: 若哈希不一致只警告不报错（因为 manythings 文件可能更新）
    strict_sha1=True:  哈希不一致直接报错
    """
    os.makedirs(cache_dir, exist_ok=True)
    zip_path = os.path.join(cache_dir, filename)

    # 你可按需增删镜像地址，按顺序尝试
    urls = [
        "http://www.manythings.org/anki/fra-eng.zip",
        "https://www.manythings.org/anki/fra-eng.zip",
    ]

    need_download = True
    if os.path.exists(zip_path):
        if expected_sha1:
            cur_sha1 = sha1sum(zip_path)
            if cur_sha1 == expected_sha1:
                need_download = False
            else:
                print(f"[警告] 本地 zip SHA1 不匹配: {cur_sha1}")
                if strict_sha1:
                    need_download = True
                else:
                    print("[提示] 将继续使用/覆盖该文件（strict_sha1=False）")
                    need_download = False
        else:
            need_download = False

    if need_download:
        ok = False
        last_err = None
        for url in urls:
            try:
                print(f"正在下载: {url}")
                urllib.request.urlretrieve(url, zip_path)
                ok = True
                break
            except Exception as e:
                last_err = e
                print(f"[失败] {url} -> {e}")
        if not ok:
            raise RuntimeError(f"下载失败，最后错误: {last_err}")

    # 哈希校验（下载后）
    if expected_sha1 and os.path.exists(zip_path):
        cur_sha1 = sha1sum(zip_path)
        if cur_sha1 != expected_sha1:
            msg = f"[警告] 下载文件 SHA1={cur_sha1}，与期望值不一致={expected_sha1}"
            if strict_sha1:
                raise ValueError(msg)
            else:
                print(msg)

    return zip_path

def extract_fra_eng(zip_path, extract_dir="./data"):
    """解压 fra-eng.zip，返回解压后的目录路径。"""
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    # 压缩包通常会解压出 ./data/fra-eng/
    return os.path.join(extract_dir, "fra-eng")

def read_data_nmt(data_dir="./data"):
    """
    兼容两种结构：
    1) ./data/fra-eng/fra.txt
    2) ./data/fra.txt
    也兼容只存在 zip 的情况（自动从 zip 读）
    """
    p1 = os.path.join(data_dir, "fra-eng", "fra.txt")
    p2 = os.path.join(data_dir, "fra.txt")
    zf = os.path.join(data_dir, "fra-eng.zip")

    # 优先读已解压文件
    if os.path.exists(p1):
        with open(p1, "r", encoding="utf-8") as f:
            return f.read()
    if os.path.exists(p2):
        with open(p2, "r", encoding="utf-8") as f:
            return f.read()

    # 没有解压文件就直接从 zip 里读
    if os.path.exists(zf):
        with zipfile.ZipFile(zf, "r") as z:
            names = z.namelist()
            # 兼容 zip 内部两种路径
            if "fra.txt" in names:
                return z.read("fra.txt").decode("utf-8")
            if "fra-eng/fra.txt" in names:
                return z.read("fra-eng/fra.txt").decode("utf-8")
            raise FileNotFoundError(f"zip 内未找到 fra.txt，实际文件: {names[:20]}")

    raise FileNotFoundError("未找到数据集文件，请检查 ./data/fra-eng.zip 或已解压的 fra.txt")


# 测试
raw_text = read_data_nmt()
print(f"len_raw_text={len(raw_text)}")
print(raw_text[:120])

# 显示文本数据的样子

Go.	Va !	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1158250 (Wittydev)
Go.	Marche.	CC-BY 2.0 (France) 


In [209]:
# 对读取出的数据进行预处理
def preprocess_nmt(text):
    """
    预处理“英语－法语”数据集
    空格代替*不间断空格*
    使用小写字母替换大写字母，
    并在单词和标点符号之间插入空格。
    """
    def no_space(char, prev_char):
        return char in set(',.!?') and prev_char != ' '

    # 使用空格替换不间断空格
    # 使用小写字母替换大写字母
    text = text.replace('\u202f', ' ').replace('\xa0', ' ').lower()
    # 在单词和标点符号之间插入空格
    out = [' ' + char if i > 0 and no_space(char, text[i - 1]) else char
        for i, char in enumerate(text)]
    return ''.join(out)

text = preprocess_nmt(raw_text)
print(text[:80])
# 显示预处理结果

go .	va !	cc-by 2 .0 (france) attribution: tatoeba .org #2877272 (cm) & #1158250


In [ ]:
# 讲一个句子，拆分为一个个 token，，
# 早期是讲一个句子拆分为一个个字符(字母)或是单词
# 目前流行的是将句子拆分为一个个“子词”---------这也是这份代码后续的更新方向和思路
def tokenize_nmt_clean(text,num_examples=1000,shuffle=True,
    seed=42,deduplicate_source=True,
    max_src_len=None,max_tgt_len=None
):
    """
    返回:
        source: List[List[str]]
        target: List[List[str]]

    特点:
    - 兼容新版 manythings 三列格式
    - 默认按 source 去重，避免一源多译干扰调试
    - 默认先打乱再截样本，避免前几百条全是 go./hi.
    - 可选限制句长
    """
    lines = text.split('\n')

    # 先收集合法样本
    pairs = []
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 2:
            src = parts[0].strip()
            tgt = parts[1].strip()
            if src and tgt:
                pairs.append((src, tgt))

    # 打乱
    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(pairs)

    # 去重：同一个英文句子只保留一个法语翻译
    if deduplicate_source:
        seen = set()
        uniq_pairs = []
        for src, tgt in pairs:
            if src not in seen:
                seen.add(src)
                uniq_pairs.append((src, tgt))
        pairs = uniq_pairs

    source, target = [], []

    for src, tgt in pairs:
        src_tokens = src.split(' ')
        tgt_tokens = tgt.split(' ')

        if max_src_len is not None and len(src_tokens) > max_src_len:
            continue
        if max_tgt_len is not None and len(tgt_tokens) > max_tgt_len:
            continue

        source.append(src_tokens)
        target.append(tgt_tokens)

        if num_examples is not None and len(source) >= num_examples:
            break

    return source, target


In [ ]:
# 将一个个词元映射为一个个唯一的数字
# 就是为词元建立索引，形成词典
class Vocab:
    """
    文本词表
    将文本转化为词元后，需要将各个词元组合到一起，建立一个词表
    """
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                reverse=True)
        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx
                            for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        """转回字符串"""
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):  # 未知词元的索引为0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def count_corpus(tokens):
    """统计词元的频率"""
    # 这里的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将词元列表展平成一个列表
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)

In [ ]:

# 填充，为了成批量处理，将所有样本的长度变成相同长度
# 所以，有的句子太短，就几个词，我们就要向句子末尾填充padding_token
# 有的句子太长，就超过num_steps，我们就将句子截断为num_steps
def truncate_pad(line, num_steps, padding_token):
    """
    截断或填充文本序列
    line: 输入文本序列,
    num_steps: 规定一个文本序列的长度为num_steps,所以我们可能需要将文本序列截断为固定长度,
                或者使用padding_token对原序列进行填充
    padding_token: 用来填充的token
    现在只是词元处理,等到很远之后会有一个Embedding层,将词元映射为向量，然后进行训练
    经过Embedding层,一个文本序列(line)就是一个矩阵,每一行表示一个词元,共num_step行,

    """
    if len(line) > num_steps:
        return line[:num_steps]  # 截断
    return line + [padding_token] * (num_steps - len(line))  # 填充



In [ ]:

# 将数据组织为一个小批量，随后训练模型时，可以一次处理多个样本，从而加速训练。
def build_array_nmt(lines, vocab, num_steps):
    """
    将机器翻译的文本序列转换成小批量
    现在我们定义一个函数，可以将文本序列,转换成小批量数据集用于训练
    我们将特定的“<eos>”词元添加到所有序列的末尾，用于表示序列的结束。
    """
    lines = [vocab[l] for l in lines]
    lines = [l + [vocab['<eos>']] for l in lines]
    array = torch.tensor([truncate_pad(
        l, num_steps, vocab['<pad>']) for l in lines])
    valid_len = (array != vocab['<pad>']).type(torch.int32).sum(1)
    return array, valid_len
# 返回结果为：(tensor([[ 44,55,43,32,21,3,1,1,1,1,1]]),5)  3是结束符号<eos>,1是填充符号<pad>
# 之所以3是结束符号<eos>,1是填充符号<pad>，
# 是因为
# src_vocab = Vocab(source, min_freq=2,
    #                     reserved_tokens=['<pad>', '<bos>', '<eos>'])
    # tgt_vocab = Vocab(target, min_freq=2,
    #                     reserved_tokens=['<pad>', '<bos>', '<eos>'])

![标签.png](../images/label.png)  
先添加结束符<eos>,随后再进行填充
但上述函数的操作结果，是数字，
输入给该函数的是一个数字列表，列表代表一个句子，列表中的数字，代表各个词元在词表中对应的数字
先在原始数字列表后面加上结束符<eos>对应的数字，在添加填充符号<padding>对应的数字

In [ ]:
# 将组织好的小批量数据集打包成 DataLoader，这样这个数据就可以使用pytorch 训练。
# 这个函数没用上
def load_array(data_arrays, batch_size, is_train=True):
    """
    打包为 DataLoader。

    参数:
    data_arrays: 多个张量组成的元组
    batch_size: 批大小
    is_train: 是否打乱
    """
    # TensorDataset:用于将多个张量（tensors）打包成一个标准的 Dataset 对象，
    # 便于与 DataLoader 配合使用。
    dataset = TensorDataset(*data_arrays)
    return DataLoader(dataset, batch_size=batch_size, shuffle=is_train)

#### 关于TensorDataset的解释
![TensorDataset注解.png](../images/TensorDataset.png)

In [ ]:
# 整合之前对于数据预处理的函数
# 完成下载文本数据集，读取文本数据集，将文本数据集词元化（就是把文本数据集进行分词，划分为一个个有意义的单元（字符，单词，子词））
# 对于源数据词元(英语)建立词元索引（词元词典），对于目标数据词元(法语)建立词元索引（词元词典）
# 组织源数据和目标数据为一个小批量，并将该小批量数据打包成 DataLoader，这样这个数据就可以使用pytorch 训练。
# 最终返回 data_iter, src_vocab, tgt_vocab, source, target
def load_data_nmt_clean(batch_size=32,num_steps=10,num_examples=1000,
    min_freq=1,shuffle=True,seed=42,deduplicate_source=True,
    max_src_len=10,max_tgt_len=10
):
    """
    返回:
        data_iter:训练数据集
        src_vocab:源语言的 Vocab 词表      一堆源语言(英语)中各个单词对应的数字
        tgt_vocab:目标语言的 Vocab 词表     一堆目标语言(法语)中各个单词对应的数字
        source:源语言数据集  
        target:目标语言数据集
    """
    # 读入文本，并对文本进行预处理（处理空格，大小写，标点符号）
    text = preprocess_nmt(read_data_nmt())
    # 对文本进行词元化
    source, target = tokenize_nmt_clean(text=text,num_examples=num_examples,shuffle=shuffle,
        seed=seed,deduplicate_source=deduplicate_source,
        max_src_len=max_src_len,max_tgt_len=max_tgt_len
    )
    # 对于源语言和目标语言分别创建一个 Vocab 词表
    # src_vocab是源语言的 Vocab 词表,tgt_vocab是目标语言的 Vocab 词表
    src_vocab = Vocab(source,min_freq=min_freq,
        reserved_tokens=['<pad>', '<bos>', '<eos>']
    )
    tgt_vocab = Vocab(target,min_freq=min_freq,
        reserved_tokens=['<pad>', '<bos>', '<eos>']
    )
    # 准备训练所需要的源数据集合目标数据集
    src_array, src_valid_len = build_array_nmt(source, src_vocab, num_steps)
    tgt_array, tgt_valid_len = build_array_nmt(target, tgt_vocab, num_steps)
    # 将源数据集合目标数据集组合成训练数据集,成批量组织,调用torch.utils.data.DataLoader()
    dataset = TensorDataset(src_array, src_valid_len, tgt_array, tgt_valid_len)
    data_iter = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    # 返回训练数据集和词表
    return data_iter, src_vocab, tgt_vocab, source, target

In [216]:
# =========================================================
# 2) 损失函数与训练工具
# =========================================================
def sequence_mask(X, valid_len, value=0):
    """
    根据 valid_len 生成掩码，覆盖无效位置。

    参数:
    X: 输入张量，常见 shape=(B, T)
    valid_len: 每个样本有效长度，shape=(B,)
    value: 无效位置填充值
    """
    maxlen = X.size(1)
    mask = torch.arange(maxlen, device=X.device)[None, :] < valid_len[:, None]
    X = X.clone()
    X[~mask] = value
    return X


class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """
    带 padding mask 的交叉熵。
    仅对有效 token 计损失。
    """
    def __init__(self):
        super().__init__(reduction='none')

    def forward(self, pred, label, dec_valid_len):
        """
        参数:
        pred: 预测 logits, shape=(batch_size, seq_len, vocab_size)
        label: 真值 id, shape=(batch_size, seq_len)
        dec_valid_len: 标签有效长度, shape=(batch_size,)

        返回:
        shape=(B,) 的样本级平均损失
        """
        weights = sequence_mask(torch.ones_like(label, dtype=torch.float32), dec_valid_len)
        unweighted = super().forward(pred.permute(0, 2, 1), label)  # -> (batch_size, seq_len)
        return (unweighted * weights).sum(dim=1) / dec_valid_len


def grad_clipping(net, theta):
    """
    梯度裁剪，避免梯度爆炸。
    参数:
    net: 模型
    theta: 裁剪阈值
    """
    params = [p for p in net.parameters() if p.grad is not None]
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for p in params:
            p.grad[:] *= theta / norm

    # 若
    # net = nn.Sequential(
    # nn.Linear(2, 3),
    # nn.ReLU(),
    # nn.Linear(3, 1)
    # )
    # 则
    # net.parameters() 包含：
    # p1: Linear1.weight → (3, 2)
    # p2: Linear1.bias   → (3)
    # p3: Linear2.weight → (1, 3)
    # p4: Linear2.bias   → (1)
    # params = [...]
    # 就是：
    # [
    # tensor(3x2),
    # tensor(3),
    # tensor(1x3),
    # tensor(1)
    # ]


# 关于梯度裁剪
🎯 一句话结论（先给你定心针）  

❌ 这个公式 不是对每个梯度向量单独裁剪  
✅ 而是对 整个梯度（拼成一个大向量）统一裁剪  
👉 这里的 g 不是一个小向量（比如某一层）  
👉 而是：  
整个模型所有参数梯度拼起来的一个大向量  
✅ 实际上：  
g = [g₁, g₂, g₃, ..., gₙ]  
👉 是：  
把所有参数梯度“拼接成一个超级长向量”  
📌 举个具体例子（最关键）  

假设模型只有两个参数：  

p1.grad = [6, 8]  
p2.grad = [3, 4]  
👉 拼成一个整体梯度：  
g = [6, 8, 3, 4]  
👉 计算 norm：  
||g|| = sqrt(36 + 64 + 9 + 16) = sqrt(125)  

## 对每个梯度向量单独裁剪为什么会改变梯度方向
![梯度裁剪改变方向的理解.png](../images/grad_clipping.png)

In [217]:
# =========================================================
# 3) 训练
# =========================================================
def train_seq2seq_minimal(net, data_iter, lr, num_epochs, tgt_vocab, device):
    """
    训练主循环（teacher forcing）。

    参数:
    net: EncoderDecoder 模型
    data_iter: 训练 DataLoader
    lr: 学习率
    num_epochs: 训练轮数
    tgt_vocab: 目标词表（取 <bos>/<pad>）
    device: 训练设备
    """
    def xavier_init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)

    net.apply(xavier_init_weights)
    net.to(device)

    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss()

    for epoch in range(num_epochs):
        net.train()
        total_loss, total_tokens = 0.0, 0

        for X, X_valid_len, Y, Y_valid_len in data_iter:
            X, X_valid_len = X.to(device), X_valid_len.to(device)
            Y, Y_valid_len = Y.to(device), Y_valid_len.to(device)

            # 构造 decoder 输入:
            # dec_input = [<bos>, y1, y2, ..., y_{T-1}]
            bos = torch.full((Y.shape[0], 1), tgt_vocab['<bos>'], dtype=torch.long, device=device)
            dec_input = torch.cat([bos, Y[:, :-1]], dim=1)
            # Y[:, :-1],每一列的最开头加了一个tag_vocab['<bos>'],
            # 所以连接时，就把原本每一列的最后一列舍去

            optimizer.zero_grad()

            # 这里调用的是你 patched 后的 net.forward
            Y_hat = net(X, dec_input, X_valid_len,Y_valid_len)

            l = loss(Y_hat, Y, Y_valid_len)
            l.sum().backward()
            grad_clipping(net, 1.0)
            optimizer.step()

            total_loss += l.sum().item()
            total_tokens += Y_valid_len.sum().item()

            with torch.no_grad():
              pred_ids = Y_hat.argmax(dim=-1)   # (B, T)
              mask = (torch.arange(Y.shape[1], device=Y.device)[None, :] < Y_valid_len[:, None])
              correct = ((pred_ids == Y) & mask).sum().item()
              total = mask.sum().item()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            # 打印损失函数和准确率
            print(f"epoch {epoch+1:>3d} | token_loss {total_loss / total_tokens:.4f}，token_acc={correct/total:.4f}")


In [218]:
# # =========================================================
# # 4) 推理（贪心解码）- 适配当前接口
# # decoder(X, valid_lens, state)
# # =========================================================
# @torch.no_grad()
# def predict_seq2seq_minimal(net, src_sentence, src_vocab, tgt_vocab, num_steps, device):
#     """
#     单句推理（greedy decoding）。

#     参数:
#     net: 训练好的模型（EncoderDecoder）
#     src_sentence: 输入源句（英文）  仅一个句子而已
#     src_vocab: 源词表
#     tgt_vocab: 目标词表
#     num_steps: 最大生成长度
#     device: 设备

#     返回:
#     str: 预测句子（空格拼接）
#     """
#     net.eval()

#     # 1) 单句预处理 + 分词（注意：这里不能用 tokenize_nmt）
#     src_sentence = preprocess_nmt(src_sentence.strip())
#     src_tokens = src_sentence.split(' ')
#     src_ids = src_vocab[src_tokens] + [src_vocab['<eos>']]

#     print("src_tokens =", src_tokens)
#     print("src_ids =", src_ids)
#     print("mapped_back =", src_vocab.to_tokens(src_ids))

#     enc_X = torch.tensor(
#         [truncate_pad(src_ids, num_steps, src_vocab['<pad>'])],
#         dtype=torch.long,
#         device=device
#     )
#     enc_valid_len = torch.tensor(
#         [(enc_X != src_vocab['<pad>']).sum().item()],
#         dtype=torch.long,
#         device=device
#     )

#     # 2) 编码 + 初始化解码状态
#     enc_outputs = net.encoder(enc_X, enc_valid_len)
#     dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)

#     # 3) 从 <bos> 开始逐步生成
#     dec_X = torch.tensor([[tgt_vocab['<bos>']]], dtype=torch.long, device=device)
#     output_ids = []

#     for _ in range(num_steps):
#         # 你的接口: decoder(X, valid_lens, state)
#         # 推理阶段逐 token 生成，valid_lens 传 None 即可
#         Y= net.decoder(dec_X, None, dec_state)

#         # 取当前步概率最大的词作为下一步输入
#         dec_X = Y.argmax(dim=2)[:, -1:]    # shape=(1,1)
#         pred = int(dec_X.item())

#         last_logits = Y[:, -1, :]
#         topk = torch.topk(last_logits, k=10, dim=1)

#         print("top10 ids =", topk.indices[0].tolist())
#         print("top10 tokens =", [tgt_vocab.to_tokens([int(i)])[0] for i in topk.indices[0]])
#         print("top10 vals =", topk.values[0].tolist())

#         # 只取最后一步概率最大的词
#         # dec_X: [batch_size, 1],他就是每一个batch的输出，也就是一个batch中，每一步的输出
#         # 之所以可以每一次for循环只使用最新的输出
#         # 是因为你在 DecoderBlock 里用了 state[2][i] 做历史 key/value 缓存。
#         #
#         # if state[2][self.i] is None:
#         #     key_values = X
#         # else:
#         #     key_values = torch.cat((state[2][self.i], X), dim=1)
#         # state[2][self.i] = key_values
#         #
#         # 经过上述代码，DecoderBlock 在于测试就可以将每次for循环的预测结果拼接起来,
#         # 在 DecoderBlock 内部就可以实现查看本次预测之前所有预测信息的能力

#         if pred == tgt_vocab['<eos>']:
#             break
#         output_ids.append(pred)

#     return ' '.join(tgt_vocab.to_tokens(output_ids))


In [219]:
@torch.no_grad()
def predict_seq2seq_minimal(net, src_sentence, src_vocab, tgt_vocab, num_steps, device):
    net.eval()

    # 1) 源句预处理
    src_sentence = preprocess_nmt(src_sentence.strip().lower())
    src_tokens = src_sentence.split(' ')
    src_ids = src_vocab[src_tokens] + [src_vocab['<eos>']]

    enc_X = torch.tensor(
        [truncate_pad(src_ids, num_steps, src_vocab['<pad>'])],
        dtype=torch.long,
        device=device
    )
    enc_valid_len = torch.tensor(
        [(enc_X != src_vocab['<pad>']).sum().item()],
        dtype=torch.long,
        device=device
    )

    # 2) 编码一次
    enc_outputs = net.encoder(enc_X, enc_valid_len)

    # 3) 从 <bos> 开始，逐步把完整前缀送进 decoder
    output_ids = []
    dec_input_ids = [tgt_vocab['<bos>']]

    for step in range(num_steps):
        dec_X = torch.tensor([dec_input_ids], dtype=torch.long, device=device)

        # 每一步都重新初始化 state，避免 cache 逻辑干扰
        dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)

        Y = net.decoder(dec_X, None, dec_state)   # shape=(1, cur_len, vocab_size)
        pred = int(Y[:, -1, :].argmax(dim=-1).item())

        # print(f"step={step}, pred_id={pred}, pred_token={tgt_vocab.to_tokens([pred])}")

        if pred == tgt_vocab['<eos>']:
            break

        output_ids.append(pred)
        dec_input_ids.append(pred)

    print("output_ids =", output_ids)
    print("output_tokens =", tgt_vocab.to_tokens(output_ids))

    return ' '.join(tgt_vocab.to_tokens(output_ids))

In [220]:
# =========================================================
# 5) 模型组装（参数顺序按你 notebook 的构造函数）
# =========================================================
def build_minimal_model(
    src_vocab_size,
    tgt_vocab_size,
    num_hiddens=64,
    num_heads=8,
    num_layers=6,
    ffn_hiddens=128,
    dropout=0.1
):
    """
    按你已有类签名构建 Encoder + Decoder + EncoderDecoder。
    """
    encoder = TransformerEncoder(
        src_vocab_size,
        num_hiddens, num_hiddens, num_hiddens,
        num_hiddens,num_heads,
        ffn_hiddens,  num_layers, dropout
    )
    decoder = TransformerDecoder(
        tgt_vocab_size,
        num_hiddens, num_hiddens, num_hiddens,
        num_hiddens, num_heads,
        ffn_hiddens,  num_layers, dropout
    )
    return EncoderDecoder(encoder, decoder)


In [221]:
# =========================================================
# 6) 运行示例
# =========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_iter, src_vocab, tgt_vocab,source, target = load_data_nmt_clean(batch_size=32, num_steps=10)
net = build_minimal_model(len(src_vocab), len(tgt_vocab))

train_seq2seq_minimal(
    net=net,
    data_iter=data_iter,
    lr=0.001,
    num_epochs=300,
    tgt_vocab=tgt_vocab,
    device=device
)



epoch   1 | token_loss 0.8785，token_acc=0.1429
epoch  10 | token_loss 0.6120，token_acc=0.3158
epoch  20 | token_loss 0.5694，token_acc=0.2667
epoch  30 | token_loss 0.5427，token_acc=0.3115
epoch  40 | token_loss 0.5096，token_acc=0.2388
epoch  50 | token_loss 0.4775，token_acc=0.2857
epoch  60 | token_loss 0.4483，token_acc=0.4286
epoch  70 | token_loss 0.4366，token_acc=0.3871
epoch  80 | token_loss 0.4003，token_acc=0.3065
epoch  90 | token_loss 0.3593，token_acc=0.3793
epoch 100 | token_loss 0.3190，token_acc=0.4348
epoch 110 | token_loss 0.2838，token_acc=0.4769
epoch 120 | token_loss 0.2498，token_acc=0.5000
epoch 130 | token_loss 0.2247，token_acc=0.5077
epoch 140 | token_loss 0.2066，token_acc=0.5246
epoch 150 | token_loss 0.1922，token_acc=0.5735
epoch 160 | token_loss 0.1829，token_acc=0.6667
epoch 170 | token_loss 0.1721，token_acc=0.7115
epoch 180 | token_loss 0.1669，token_acc=0.6400
epoch 190 | token_loss 0.1642，token_acc=0.6852
epoch 200 | token_loss 0.1600，token_acc=0.7581
epoch 210 | t

In [ ]:
# 观察部分源语言和目标语言，确保的代码正确读入了数据
print("样本数:", len(source))
print("source[:5] =", source[:5])
print("target[:5] =", target[:5])
print("src_vocab size =", len(src_vocab))
print("tgt_vocab size =", len(tgt_vocab))

样本数: 1000
source[:5] = [['can', 'he', 'see', 'me', '?'], ['i', "didn't", 'know', 'you', 'had', 'company', '.'], ['tom', 'walks', 'quickly', '.'], ["let's", 'do', 'this', 'first', 'of', 'all', '.'], ['i', "didn't", 'know', 'that', 'it', 'was', 'there', '.']]
target[:5] = [['est-il', 'en', 'mesure', 'de', 'me', 'voir', '?'], ["j'ignorais", 'que', 'tu', 'avais', 'de', 'la', 'compagnie', '.'], ['tom', 'marche', 'rapidement', '.'], ['faisons', "d'abord", 'ceci', '.'], ["j'ignorais", "qu'il", "s'y", 'trouvait', '.']]
src_vocab size = 1295
tgt_vocab size = 1714


In [223]:
# 观察数据集中的前几个样本
# 可以用这些样本来证明模型是是否具备死记硬背的能力
for i in range(10):
    print(i, ' '.join(source[i]), '->', ' '.join(target[i]))

0 can he see me ? -> est-il en mesure de me voir ?
1 i didn't know you had company . -> j'ignorais que tu avais de la compagnie .
2 tom walks quickly . -> tom marche rapidement .
3 let's do this first of all . -> faisons d'abord ceci .
4 i didn't know that it was there . -> j'ignorais qu'il s'y trouvait .
5 we can't wait for the weekend . -> nous sommes impatients d'être en week-end .
6 i got injured in the traffic accident . -> j'ai été blessé dans l'accident de la route .
7 would you teach me ? -> me l'enseignerais-tu ?
8 that was the basic idea . -> c'était l'idée de base .
9 are you in a safe place now ? -> êtes-vous à présent en lieu sûr ?


In [224]:
# 使用训练集中的数据集做预测
# 确定模型具备初级的死记硬背的能力，模型要是有了这个能力就说明模型可以学习，之后再用其他数据集测模型的泛化能力
# 也可以用这个数据集来调整修改代码的逻辑逻辑错误
for i in range(3):
    src_sent = ' '.join(source[i])
    pred = predict_seq2seq_minimal(
        net=net,
        src_sentence=src_sent,
        src_vocab=src_vocab,
        tgt_vocab=tgt_vocab,
        num_steps=10,
        device=device
    )
    print(f"src:  {src_sent}")
    print(f"gold: {' '.join(target[i])}")
    print(f"pred: {pred}")
    print('-' * 50)

output_ids = [5, 14, 30, 8, 810, 4]
output_tokens = ['je', 'ne', 'suis', 'pas', 'contente', '.']
src:  can he see me ?
gold: est-il en mesure de me voir ?
pred: je ne suis pas contente .
--------------------------------------------------
output_ids = [5, 14, 30, 8, 810, 4]
output_tokens = ['je', 'ne', 'suis', 'pas', 'contente', '.']
src:  i didn't know you had company .
gold: j'ignorais que tu avais de la compagnie .
pred: je ne suis pas contente .
--------------------------------------------------
output_ids = [5, 14, 30, 8, 810, 4]
output_tokens = ['je', 'ne', 'suis', 'pas', 'contente', '.']
src:  tom walks quickly .
gold: tom marche rapidement .
pred: je ne suis pas contente .
--------------------------------------------------


### 打印输入映射：如果里面有很多 <unk>，那这个测试本身就偏苛刻。
### 不适合我这个小模型
### 预测数据的类型

### 第一类:训练集句子---证明模型具备学习的能力，死记硬背也算是在学习

### 第二类:轻微改写句子---证明模型具备一定的泛化能力
### i didn't know you had company .
### i didn't know you were here .
### tom walks quickly .
### tom runs quickly .

### 第三类:完全新句子---证明模型具备泛化能力
### i am a student



In [225]:
# 批量测试函数（带调试信息）
@torch.no_grad()
def test_sentences(net, sentences, src_vocab, tgt_vocab, num_steps, device):
    net.eval()

    for raw_sentence in sentences:
        print("🔹 raw_sentence =", raw_sentence)

        # ====== 1. 预处理 ======
        processed = preprocess_nmt(raw_sentence.strip())
        src_tokens = processed.split(' ')
        src_ids = src_vocab[src_tokens] + [src_vocab['<eos>']]

        print("processed =", processed)
        print("src_tokens =", src_tokens)
        print("src_ids =", src_ids)
        print("mapped_back =", src_vocab.to_tokens(src_ids))

        # ====== 2. 检查是否有 <unk> ======
        if src_vocab['<unk>'] in src_ids:
            print("⚠️  含有 <unk>，语义可能丢失")

        # ====== 3. 推理 ======
        pred = predict_seq2seq_minimal(
            net=net,
            src_sentence=raw_sentence,   # ⚠️ 用原句，不用 processed
            src_vocab=src_vocab,
            tgt_vocab=tgt_vocab,
            num_steps=num_steps,
            device=device
        )

        print("pred =", pred)
        print("=" * 40)

In [226]:
test_sentences(
    net,
    [
        "i didn't know you had company .",
        "i am a student",
        "tom walks quickly .",
        "are you in a safe place now ?"
    ],
    src_vocab,
    tgt_vocab,
    num_steps=10,
    device=device
)

🔹 raw_sentence = i didn't know you had company .
processed = i didn't know you had company .
src_tokens = ['i', "didn't", 'know', 'you', 'had', 'company', '.']
src_ids = [5, 45, 42, 6, 57, 341, 4, 3]
mapped_back = ['i', "didn't", 'know', 'you', 'had', 'company', '.', '<eos>']
output_ids = [5, 14, 30, 8, 810, 4]
output_tokens = ['je', 'ne', 'suis', 'pas', 'contente', '.']
pred = je ne suis pas contente .
🔹 raw_sentence = i am a student
processed = i am a student
src_tokens = ['i', 'am', 'a', 'student']
src_ids = [5, 139, 10, 0, 3]
mapped_back = ['i', 'am', 'a', '<unk>', '<eos>']
⚠️  含有 <unk>，语义可能丢失
output_ids = [5, 14, 30, 8, 810, 4]
output_tokens = ['je', 'ne', 'suis', 'pas', 'contente', '.']
pred = je ne suis pas contente .
🔹 raw_sentence = tom walks quickly .
processed = tom walks quickly .
src_tokens = ['tom', 'walks', 'quickly', '.']
src_ids = [12, 342, 251, 4, 3]
mapped_back = ['tom', 'walks', 'quickly', '.', '<eos>']
output_ids = [5, 14, 30, 8, 810, 4]
output_tokens = ['je', 'ne'

In [227]:
# 比较同一输入在训练和推理状态下的输出
# 目的是要明确推理的逻辑是否出问题
# 如果训练与推理输出一致，说明，推理逻辑没问题
# 如果不一致，说明推理逻辑有问题
# 所用的测试数据随意，建议使用训练时使用的数据集
@torch.no_grad()
def compare_train_eval_next_token(net, src_tokens, tgt_prefix_tokens,
                                  src_vocab, tgt_vocab, num_steps, device):
    # 编码
    src_ids = src_vocab[src_tokens] + [src_vocab['<eos>']]
    enc_X = torch.tensor(
        [truncate_pad(src_ids, num_steps, src_vocab['<pad>'])],
        dtype=torch.long, device=device
    )
    enc_valid_len = torch.tensor(
        [(enc_X != src_vocab['<pad>']).sum().item()],
        dtype=torch.long, device=device
    )

    enc_outputs = net.encoder(enc_X, enc_valid_len)

    dec_ids = [tgt_vocab['<bos>']] + tgt_vocab[tgt_prefix_tokens]
    dec_X = torch.tensor([dec_ids], dtype=torch.long, device=device)
    dec_valid_len = torch.tensor([len(dec_ids)], dtype=torch.long, device=device)

    # eval 模式
    net.eval()
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    Y_eval = net.decoder(dec_X, None, dec_state)
    eval_topk = torch.topk(Y_eval[:, -1, :], k=5, dim=-1)

    # train 模式（只为对比输出，不做反传）
    net.train()
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    Y_train = net.decoder(dec_X, dec_valid_len, dec_state)
    train_topk = torch.topk(Y_train[:, -1, :], k=5, dim=-1)

    net.eval()

    print("prefix:", tgt_prefix_tokens)
    print("eval top5 tokens:",
          [tgt_vocab.to_tokens([int(i)])[0] for i in eval_topk.indices[0]])
    print("eval top5 vals:",
          [float(v) for v in eval_topk.values[0]])

    print("train top5 tokens:",
          [tgt_vocab.to_tokens([int(i)])[0] for i in train_topk.indices[0]])
    print("train top5 vals:",
          [float(v) for v in train_topk.values[0]])

In [228]:
compare_train_eval_next_token(
    net,
    source[1],                # 比如 i didn't know you had company .
    ["j'ignorais"],           # 只给前缀
    src_vocab,
    tgt_vocab,
    num_steps=10,
    device=device
)

prefix: ["j'ignorais"]
eval top5 tokens: ["qu'il", 'que', 'qui', 'bien', 'est']
eval top5 vals: [16.36334991455078, 13.889881134033203, 11.022472381591797, 10.035630226135254, 9.657583236694336]
train top5 tokens: ["qu'il", 'que', 'qui', ',', 'bien']
train top5 vals: [14.316110610961914, 12.964532852172852, 10.921842575073242, 9.399922370910645, 8.90537166595459]


In [229]:
@torch.no_grad()
def debug_with_gold_prefix(net, src_tokens, tgt_tokens, src_vocab, tgt_vocab, num_steps, device):
    net.eval()

    src_ids = src_vocab[src_tokens] + [src_vocab['<eos>']]
    enc_X = torch.tensor(
        [truncate_pad(src_ids, num_steps, src_vocab['<pad>'])],
        dtype=torch.long,
        device=device
    )
    enc_valid_len = torch.tensor(
        [(enc_X != src_vocab['<pad>']).sum().item()],
        dtype=torch.long,
        device=device
    )

    enc_outputs = net.encoder(enc_X, enc_valid_len)

    gold_ids = tgt_vocab[tgt_tokens] + [tgt_vocab['<eos>']]
    prefix = [tgt_vocab['<bos>']]

    for step in range(min(len(gold_ids), num_steps)):
        dec_X = torch.tensor([prefix], dtype=torch.long, device=device)
        dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
        Y = net.decoder(dec_X, None, dec_state)

        pred = int(Y[:, -1, :].argmax(dim=-1).item())
        gold = gold_ids[step]

        print(
            f"step={step} | pred={tgt_vocab.to_tokens([pred])} | gold={tgt_vocab.to_tokens([gold])}"
        )

        prefix.append(gold)

In [230]:
debug_with_gold_prefix(
    net,
    source[1],
    target[1],
    src_vocab,
    tgt_vocab,
    num_steps=10,
    device=device
)

step=0 | pred=['je'] | gold=["j'ignorais"]
step=1 | pred=["qu'il"] | gold=['que']
step=2 | pred=['tu'] | gold=['tu']
step=3 | pred=['avais'] | gold=['avais']


step=4 | pred=['de'] | gold=['de']
step=5 | pred=['la'] | gold=['la']
step=6 | pred=['compagnie'] | gold=['compagnie']
step=7 | pred=['.'] | gold=['.']
step=8 | pred=['<eos>'] | gold=['<eos>']
